In [4]:
import pandas as pd
import numpy as np

# 1. LOAD DATASET RITEL GLOBAL (Superstore Dataset)
urls = [
    "https://raw.githubusercontent.com/curran/data/gh-pages/superstoreSales/superstoreSales.csv",
    "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"
]

df = None
for url in urls:
    try:
        df = pd.read_csv(url, encoding='latin1')
        print(f"✅ Berhasil memuat dataset dari: {url}")
        break
    except Exception:
        continue

# Cadangan otomatis jika link eksternal terblokir
if df is None:
    print("ℹ️ Membuat dataset ritel global terstruktur (10,000 transaksi)...")
    np.random.seed(42)
    n = 10000
    dates = pd.date_range('2024-01-01', '2026-10-31', freq='h')
    order_dates = np.random.choice(dates, size=n)
    regions = np.random.choice(['West', 'East', 'Central', 'South'], size=n, p=[0.32, 0.28, 0.23, 0.17])
    segments = np.random.choice(['Consumer', 'Corporate', 'Home Office'], size=n, p=[0.52, 0.30, 0.18])
    categories = np.random.choice(['Technology', 'Office Supplies', 'Furniture'], size=n, p=[0.20, 0.60, 0.20])

    sub_cats = []
    for c in categories:
        if c == 'Technology':
            sub_cats.append(np.random.choice(['Phones', 'Machines', 'Accessories', 'Copiers']))
        elif c == 'Furniture':
            sub_cats.append(np.random.choice(['Chairs', 'Tables', 'Bookcases', 'Furnishings']))
        else:
            sub_cats.append(np.random.choice(['Binders', 'Paper', 'Storage', 'Art', 'Appliances']))

    sales = np.round(np.random.lognormal(mean=4.8, sigma=1.1, size=n), 2)
    discounts = np.random.choice([0.0, 0.1, 0.2, 0.4, 0.6], size=n, p=[0.45, 0.15, 0.25, 0.10, 0.05])

    # Simulasi fenomena bisnis nyata: Diskon >= 40% (terutama di Furniture/Tables) menyebabkan rugi!
    profits = []
    for s, d, c, sc in zip(sales, discounts, categories, sub_cats):
        margin = 0.22 if c == 'Technology' else (0.18 if c == 'Office Supplies' else 0.08)
        if sc == 'Tables':
            margin = -0.05
        profit = s * (margin - (d * 0.85)) + np.random.normal(0, 15)
        profits.append(round(profit, 2))

    df = pd.DataFrame({
        'Order_ID': [f"ORD-{20240000+i}" for i in range(n)],
        'Order_Date': pd.to_datetime(order_dates).strftime('%Y-%m-%d'),
        'Region': regions,
        'Segment': segments,
        'Category': categories,
        'Sub_Category': sub_cats,
        'Sales': sales,
        'Discount': discounts,
        'Profit': profits
    })

# Pastikan kolom tanggal & margin bersih untuk BI Tool
df.columns = [c.strip().replace(' ', '_').replace('-', '_') for c in df.columns]
if 'Profit_Margin' not in df.columns and 'Profit' in df.columns and 'Sales' in df.columns:
    df['Profit_Margin'] = np.round(df['Profit'] / df['Sales'], 4)

df.to_csv('superstore_clean_bi.csv', index=False)
print("✅ File 'superstore_clean_bi.csv' siap di-download! Total baris:", len(df))
display(df.head())

✅ Berhasil memuat dataset dari: https://raw.githubusercontent.com/curran/data/gh-pages/superstoreSales/superstoreSales.csv
✅ File 'superstore_clean_bi.csv' siap di-download! Total baris: 8399


,Row_ID,Order_ID,Order_Date,Order_Priority,Order_Quantity,Sales,Discount,Ship_Mode,Profit,Unit_Price,...,Province,Region,Customer_Segment,Product_Category,Product_Sub_Category,Product_Name,Product_Container,Product_Base_Margin,Ship_Date,Profit_Margin
0,1,3,10/13/2010,Low,6,261.5400,0.04,Regular Air,-213.25,38.94,...,Nunavut,Nunavut,Small Business,Office Supplies,Storage & Organization,"Eldon Base for stackable storage shelf, platinum",Large Box,0.80,10/20/2010,-0.8154
1,49,293,10/1/2012,High,49,10123.0200,0.07,Delivery Truck,457.81,208.16,...,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,"1.7 Cubic Foot Compact ""Cube"" Office Refrigera...",Jumbo Drum,0.58,10/2/2012,0.0452
2,50,293,10/1/2012,High,27,244.5700,0.01,Regular Air,46.71,8.69,...,Nunavut,Nunavut,Consumer,Office Supplies,Binders and Binder Accessories,"Cardinal Slant-D¨ Ring Binder, Heavy Gauge Vinyl",Small Box,0.39,10/3/2012,0.1910
3,80,483,7/10/2011,High,30,4965.7595,0.08,Regular Air,1198.97,195.99,...,Nunavut,Nunavut,Corporate,Technology,Telephones and Communication,R380,Small Box,0.58,7/12/2011,0.2414
4,85,515,8/28/2010,Not Specified,19,394.2700,0.08,Regular Air,30.94,21.78,...,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,Holmes HEPA Air Purifier,Medium Box,0.50,8/30/2010,0.0785
